The goal of this notebook is to explain the logic and math behind the CatBoost algorithm.

CatBoost is similar to XGBoost and LightGBM in terms of the loss function, Taylor approximation,
and the general boosting idea — each tree corrects the residuals of the previous one. However,
CatBoost differs in two key ways:

- **Symmetric trees** — every node at the same depth level shares the same split condition
(feature + threshold), unlike XGBoost and LightGBM where each node finds its own best split.
This symmetry means a tree of depth N has exactly 2^N leaves and can be stored as just two arrays:
one of length N holding the split conditions per level, and one of length 2^N holding the leaf outputs.
Inference requires no recursion — just N comparisons and a lookup into the leaf array.

- **Ordered Target Encoding** — CatBoost handles categorical features natively, without any
preprocessing. The encoding is computed on the fly during training in a way that prevents data leakage,
which we will cover in detail below.

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

In [ ]:
def _get_target_encoding_per_permutation(
            self,
            X: pd.DataFrame,
            y: np.ndarray,
            categorical_features: list[str],
            permutations: list[list[int]],
            global_mean: float
    ) -> None:
        for p in permutations:
            running_sum = defaultdict(dict)     # feature: threshold: (running sum, count)
            X_shuffled = X.iloc[p].copy().reset_index(drop=True)
            y_shuffled = y[p]

            for i, row in X_shuffled.iterrows():
                for feature in categorical_features:
                    threshold = row[feature]
                    # if threshold for curr category is met for the 1st time - assign the global mean to it
                    if threshold not in running_sum[feature]:
                        X_shuffled.at[i, feature] = global_mean
                        running_sum[feature][threshold] = (y_shuffled[i], 1)
                    else:
                        curr_sum, curr_count = running_sum[feature][threshold]
                        # the core idea is to calculate the mean for that threshold without curr value
                        # prevents data leakage
                        X_shuffled.at[i, feature] = (curr_sum + self.alpha * global_mean) / (curr_count + self.alpha)

                        # update running sum and count and store for curr feature/threshold pair
                        curr_sum += y_shuffled[i]
                        curr_count += 1
                        running_sum[feature][threshold] = (curr_sum, curr_count)

            self.permutations_of_X.append(X_shuffled)

As you can see in `_get_target_encoding_per_permutation`, the encoding for each row is computed
using only the rows that came before it in the current permutation — the current row's target
is never included in its own encoding. This is the core idea that prevents leakage.

When a category is seen for the first time, there is no prior data to compute a mean from,
so we fall back to the **global mean** of the target. For subsequent occurrences, we compute
a smoothed running mean:

$$\text{encoding} = \frac{\text{running\_sum} + \alpha \cdot \text{global\_mean}}{\text{count} + \alpha}$$

The **alpha** parameter controls how strongly rare categories are pulled toward the global mean.
A high alpha means even frequent categories stay close to the global mean — useful when category
counts are low and estimates are noisy.

You may have noticed that the encoding is computed not once, but across multiple random permutations
of the data. Here is why.

The ordered encoding is sensitive to the order in which rows are processed. A row that appears
early in the permutation will have a noisy encoding — very few previous rows of the same category
have been seen yet. A row that appears late will have a stable, accurate encoding. If we used a
single fixed ordering, some rows would always get noisy encodings and others would always get
accurate ones — introducing a systematic bias into training.

To fix this, CatBoost generates **N permutations** (typically 4) upfront before training starts.
Each permutation produces a differently encoded version of X. During the boosting loop, each tree
is assigned one of these permutations in a round-robin fashion — tree 1 uses permutation 1,
tree 2 uses permutation 2, and so on. This way, no row is systematically disadvantaged by its
position in a single ordering, and the variance from any one permutation is averaged out across trees.

## Symmetric Trees

Standard decision trees (used in XGBoost and LightGBM) grow asymmetrically — at each node,
the algorithm independently finds the best split for that node. This means different branches
of the tree can split on completely different features at the same depth level, giving the tree
a lot of flexibility but also making it prone to overfitting and slower at inference.

CatBoost takes a different approach. At each depth level, **every node shares the same split
condition** — the same feature and the same threshold. This is called a **symmetric** or
**oblivious** tree. The tree at depth N always has exactly 2^N leaves, and the entire structure
can be represented as:

- A list of N `(feature, threshold)` tuples — one per level
- A flat array of 2^N leaf values

This makes inference extremely fast — no recursion, just N comparisons and a single array lookup.

### Finding the Best Split

In a standard tree, gain is computed independently per node and the best split is chosen
per node. In a symmetric tree, we need one split that works best **across all nodes at the
current level**.

To achieve this, for every candidate `(feature, threshold)` pair we sum the gain across
every node at the current level. The pair with the highest **total gain** across all nodes
is chosen as the level's split. This means the chosen split may not be locally optimal for
every individual node, but it is the best global choice for the level as a whole.

In [ ]:
def _best_split(
        self, 
        X: pd.DataFrame, 
        gradients: list[np.ndarray], 
        features: list[str], 
        used_features: set[str]
        ) -> tuple[str, str | int | float]:
        best_gain = 0.0
        best_feature = None
        best_threshold = None

        for feature in features:
            is_numeric = pd.api.types.is_numeric_dtype(X[feature])

            # there is no point at reusing categorical feature since the data was already splitted based on it 
            if not is_numeric and feature in used_features:
                continue

            thresholds = X[feature].unique()
            thresholds.sort()

            # if have the numeric feature we should check the midpoints
            # midpoint = (x[i] + x[i - 1]) / 2
            if is_numeric:
                thresholds = thresholds[0:len(thresholds) - 1] + thresholds[1:]
                thresholds = thresholds / 2

            for threshold in thresholds:
                # for each threshold compute the sum of gains of every node in case we split by this threshold
                curr_total_gain = 0.0
                for i, idx in enumerate(self.partitions):
                    mask = X[feature].iloc[idx] <= threshold if is_numeric else X[feature].iloc[idx] == threshold
                    left_g = gradients[i][mask]
                    right_g = gradients[i][~mask]

                    gain = self._calculate_gain(
                        parent_ss=self._calculate_similarity_score(gradients=gradients[i]),
                        left_child_ss=self._calculate_similarity_score(gradients=left_g),
                        right_child_ss=self._calculate_similarity_score(gradients=right_g)
                    )
                    curr_total_gain += gain

                if curr_total_gain > best_gain:
                    best_gain = curr_total_gain
                    best_feature = feature
                    best_threshold = threshold
                    

        return best_feature, best_threshold

### Building the Tree Level by Level

Because the tree is symmetric, we do not build it recursively. Instead we maintain two
parallel lists that grow level by level:

- `partitions` — a list of index arrays, one per node, tracking which rows of X belong to that node
- `gradients` — a list of gradient arrays, one per node, tracking the gradients of those rows

At the root level both lists have a single entry covering all rows. After each split, every
node produces a left child and a right child — so both lists double in length at every level.
At depth D we have 2^D nodes, each with its own subset of rows and gradients.

Once `max_depth` is reached or no valid split is found, the leaf values are computed from
the final gradient arrays.

In [ ]:
def _build_tree(
        self,
        X: pd.DataFrame,
        gradients: np.ndarray,
        features: list[str],
        used_features: set[str],
    ):
        # we start from 1x1 matrices since the root is not splitted yet
        gradients = [gradients]
        self.partitions = [np.arange(len(X))]
        for _ in range(self.max_depth):
            feature, threshold = self._best_split(X, gradients, features, used_features)

            # if no children ss > parent ss for any features - new level cannot be created
            if feature is None:
                self._leaf_values(gradients=gradients)
                return 

            is_numeric = pd.api.types.is_numeric_dtype(X[feature])
            if not is_numeric:
                used_features.add(feature)
            self.splits.append((feature, threshold))

            # we need to track gradients and indices for every node 
            # because we do not build tree recursively and work on the whole X all the time
            new_partitions = []
            new_gradients = []
            for i, idx in enumerate(self.partitions):
                mask = X[feature].iloc[idx] <= threshold if is_numeric else X[feature].iloc[idx] == threshold
                left_idx = idx[mask]
                right_idx = idx[~mask]
                new_partitions.append(left_idx)
                new_partitions.append(right_idx)

                left_grad = gradients[i][mask]
                right_grad = gradients[i][~mask]
                new_gradients.append(left_grad)
                new_gradients.append(right_grad)

            gradients = new_gradients
            self.partitions = new_partitions

        self._leaf_values(gradients=gradients)
        return

### Computing Leaf Values

Once the tree stops growing, each leaf is assigned an output value. The formula is the
standard gradient boosting leaf value — the negative mean of the gradients in that leaf,
regularized by lambda:

$$\text{leaf} = -\frac{\sum \text{gradients}}{\text{count} + \lambda}$$

The negative sign is because gradients point in the direction of increasing loss — we want
to step in the opposite direction. Lambda prevents large leaf values when a leaf contains
very few samples, acting as regularization. Empty leaves — which can occur in symmetric
trees when a split sends all rows to one side — are assigned 0.

In [ ]:
def _leaf_values(self, gradients: list[np.ndarray]) -> list[float]:
        for grad in gradients:
            if len(grad) == 0:
                self.leaf_values.append(0.0)
            else:
                self.leaf_values.append(-1 * (np.sum(grad) / (grad.shape[0] + self.lam)))


### Inference — The Lookup Table Trick

Because the tree is symmetric, inference does not require traversing a tree structure.
Instead, we exploit the fact that at each level exactly half the leaf candidates correspond
to the left branch and half to the right.

We start with the full array of 2^N leaf values. At each level we evaluate the split
condition for the sample — if it goes left we keep the first half of the candidates, if
it goes right we keep the second half. After N levels we are left with exactly one value —
the leaf output for that sample.

This is O(depth) with no pointer chasing or recursion — just array slicing.

In [ ]:
def _predict_sample(self, sample: pd.Series) -> float:
        candidates = self.leaf_values

        for feature, threshold in self.splits:
            mid = len(candidates) // 2
            
            is_categorical = isinstance(threshold, str)

            on_the_left = sample[feature] == threshold if is_categorical else sample[feature] <= threshold

            if on_the_left:
                candidates = candidates[:mid]
            else:
                candidates = candidates[mid:]

        return candidates[0]

Together these four methods define a complete symmetric tree — built level by level with
a shared split per level, stored as two flat arrays, and evaluated in O(depth) at inference.

## Summary

In this notebook we covered the two core ideas that make CatBoost different from other
gradient boosting frameworks:

- **Ordered Target Encoding** — categorical features are encoded using a smoothed running
mean of previous rows in a random permutation, ensuring no row ever leaks its own target
into its own encoding. Multiple permutations are generated upfront and reused across trees
to reduce variance from any single ordering.

- **Symmetric Trees** — every node at the same depth level shares the same split condition,
making the tree representable as two flat arrays and inference reducible to N comparisons
and a single array lookup.

The full implementation — including `CatBoostTree` and `CatBoostRegressor` — is available
in the repository(./wrapped_models/). The regressor ties everything together: generating permutations upfront,
encoding categorical features per permutation, assigning each tree its own encoded X, and
accumulating predictions in original row order throughout the boosting loop.